In [0]:
%pip install fitparse

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def format_time(seconds: float) -> str:
    """Hulpfunctie om seconden om te zetten naar hh:mm:ss of mm:ss."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    if hours > 0:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"


def analyze_heartraterecovery(
    df: pd.DataFrame, 
    min_stop_sec: int = 180,    # Minimaal 3 minuten pauze voor rood pauzevlak
    min_hr_drop: int = 5,       # Toon alleen HRR-annotatie als herstel >= 5 bpm is
    hr_drop_window: int = 60, 
    hr_smooth_window: int = 60
):
    apply_triathlon_style()
    
    df_clean = df.copy().sort_values('timestamp').reset_index(drop=True)
    
    # 1. Tijdsberekening
    if 'timestamp' in df_clean.columns:
        df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
        df_clean['time_diff_sec'] = df_clean['timestamp'].diff().dt.total_seconds().fillna(0)

        start_time = df_clean['timestamp'].iloc[0]
        df_clean['active_elapsed_minutes'] = (df_clean['timestamp'] - start_time).dt.total_seconds() / 60.0
    elif 'elapsed_time' in df_clean.columns:
        df_clean['active_elapsed_minutes'] = df_clean['elapsed_time'] / 60.0
    elif 'elapsed_seconds' in df_clean.columns:
        df_clean['active_elapsed_minutes'] = df_clean['elapsed_seconds'] / 60.0
    else:
        df_clean['active_elapsed_minutes'] = df_clean['time_diff'].cumsum() / 60.0

    x_col = 'active_elapsed_minutes'

    # 2. Bepalen hoogtemeters
    alt_col = None
    for col in ['altitude', 'enhanced_altitude', 'altitude_m']:
        if col in df_clean.columns and df_clean[col].notna().any():
            alt_col = col
            break
            
    if alt_col:
        df_clean['alt_clean'] = df_clean[alt_col].interpolate(method='linear').ffill().bfill()
    
    # 3. Pauze-detectie (Snelheid/Status)
    speed_col = 'speed' if 'speed' in df_clean.columns else 'enhanced_speed'
    
    if 'is_paused' in df_clean.columns:
        df_clean['is_static'] = df_clean['is_paused']
    elif 'is_moving' in df_clean.columns:
        df_clean['is_static'] = ~df_clean['is_moving']
    elif speed_col in df_clean.columns:
        df_clean['is_static'] = df_clean[speed_col] < 0.45
    else:
        df_clean['is_static'] = False

    df_clean['pause_id'] = (df_clean['is_static'] != df_clean['is_static'].shift()).cumsum()

    # 4. Smooth hartslag & 'bevroren' blokken rond minuut 200 opschonen
    hr_col = 'heart_rate' if 'heart_rate' in df_clean.columns else 'hr'
    if hr_col in df_clean.columns:
        df_clean['hr_plot'] = df_clean[hr_col].copy()
        
        # A. Zet stilstaande stilstandsmomenten op NaN
        df_clean.loc[df_clean['is_static'], 'hr_plot'] = np.nan
        
        # B. Detecteer 'bevroren' constante hartslag (30+ datapunten exact gelijk)
        frozen_hr = (df_clean[hr_col] == df_clean[hr_col].shift(1))
        frozen_streak = frozen_hr.groupby((~frozen_hr).cumsum()).cumsum()
        df_clean.loc[frozen_streak > 30, 'hr_plot'] = np.nan

        # C. Rolling mean berekenen met soepele vensterinstelling
        df_clean['hr_smooth'] = df_clean['hr_plot'].rolling(window=hr_smooth_window, min_periods=10).mean()

    # 5. Pauze-statistieken verzamelen voor HRR-annotaties
    pauses = []
    if 'timestamp' in df_clean.columns:
        for pid, group in df_clean[df_clean['is_static']].groupby('pause_id'):
            duration = (group['timestamp'].iloc[-1] - group['timestamp'].iloc[0]).total_seconds()
            if duration >= min_stop_sec:
                t_start = group['timestamp'].iloc[0]
                hr_start = group[hr_col].iloc[0] if hr_col in group.columns else 0
                
                target_time = t_start + pd.Timedelta(seconds=hr_drop_window)
                sub_window = group[group['timestamp'] <= target_time]
                
                if not sub_window.empty and duration >= hr_drop_window:
                    hr_after_window = sub_window[hr_col].iloc[-1]
                    hrr_val = hr_start - hr_after_window
                else:
                    hrr_val = hr_start - group[hr_col].min()
                
                if hrr_val >= min_hr_drop:
                    pauses.append({
                        'start_time': t_start,
                        'x_val': group[x_col].iloc[0],
                        'duration_min': duration / 60.0,
                        'hr_start': hr_start,
                        'hrr_60s': hrr_val
                    })
            
    df_pauses = pd.DataFrame(pauses)

    # 6. Grafiek maken
    fig, ax_hr = plt.subplots(figsize=(13.33, 7.5), dpi=300)

    COLOR_ELEVATION = '#8A99AD'
    COLOR_HR        = '#FF2A6D'
    COLOR_PAUSE     = '#EF4444'

    # A. Hoogteprofiel
    if alt_col:
        ax_alt = ax_hr.twinx()
        min_alt = df_clean['alt_clean'].min()
        max_alt = df_clean['alt_clean'].max()

        ax_alt.fill_between(
            df_clean[x_col], df_clean['alt_clean'], min_alt, 
            color=COLOR_ELEVATION, alpha=0.20, zorder=1, label='Hoogteprofiel'
        )
        ax_alt.plot(df_clean[x_col], df_clean['alt_clean'], color=COLOR_ELEVATION, linewidth=1.2, alpha=0.4, zorder=1)

        ax_alt.set_ylim(min_alt, max_alt + (max_alt - min_alt) * 0.4)
        ax_alt.set_yticks([])              
        ax_alt.yaxis.set_visible(False)
        ax_alt.grid(False)

    # B. Hartslag
    if hr_col in df_clean.columns:
        ax_hr.set_zorder(10)
        ax_hr.patch.set_visible(False)

        ax_hr.plot(df_clean[x_col], df_clean['hr_smooth'], color=COLOR_HR, linewidth=1.8, label='Hartslag (BPM)', zorder=12)
        ax_hr.set_ylabel('Hartslag (BPM)', color=COLOR_HR, fontweight='bold')
        ax_hr.tick_params(axis='y', labelcolor=COLOR_HR)

        hr_data = df_clean[hr_col].dropna()
        if not hr_data.empty:
            ax_hr.set_ylim(int(hr_data.min() - 10), int(hr_data.max() + 15))

    # C. Gefilterde pauzevlakken & HRR-annotaties (alleen pauzes >= min_stop_sec)
    has_pause_label = False
    
    if not df_pauses.empty:
        for idx, row in df_pauses.iterrows():
            if row['duration_min'] >= (min_stop_sec / 60.0):
                ax_hr.axvspan(
                    row['x_val'], 
                    row['x_val'] + row['duration_min'], 
                    color=COLOR_PAUSE, 
                    alpha=0.18, 
                    zorder=11, 
                    label='Rustpauze' if not has_pause_label else ""
                )
                has_pause_label = True
            
            ax_hr.annotate(
                f"-{int(row['hrr_60s'])} bpm", 
                xy=(row['x_val'], row['hr_start']),
                xytext=(row['x_val'], row['hr_start'] + 6),
                fontsize=8.5, fontweight='bold', color='#FFFFFF',
                ha='center', zorder=13,
                arrowprops=dict(arrowstyle='->', color=COLOR_PAUSE, lw=1.2)
            )

    # D. Berekening statistieken voor overlay
    if 'timestamp' in df_clean.columns:
        elapsed_sec = (df_clean['timestamp'].iloc[-1] - df_clean['timestamp'].iloc[0]).total_seconds()
        moving_sec = df_clean[~df_clean['is_static'] & df_clean[hr_col].notna()]['time_diff_sec'].sum()
    else:
        elapsed_sec = (df_clean[x_col].iloc[-1] - df_clean[x_col].iloc[0]) * 60
        moving_sec = elapsed_sec

    elapsed_str = format_time(elapsed_sec)
    moving_str = format_time(moving_sec)

    avg_hr = int(df_clean[hr_col].mean()) if hr_col in df_clean.columns else 0
    max_hr = int(df_clean[hr_col].max()) if hr_col in df_clean.columns else 0

    stats_text = (
        f"Bewogen tijd:    {moving_str}\n"
        f"Verstreken tijd: {elapsed_str}\n"
        f"Gem. hartslag:   {avg_hr} bpm\n"
        f"Max. hartslag:   {max_hr} bpm"
    )

    # E. Overlay & Legenda
    ax_hr.text(
        0.03, 0.93, stats_text,
        transform=ax_hr.transAxes, 
        fontsize=9.5, 
        fontfamily='monospace',
        color='white', 
        fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#111827', alpha=0.85, edgecolor='#374151'),
        zorder=20
    )

    ax_hr.legend(
        loc='upper right', 
        facecolor='#111827', 
        edgecolor='#374151', 
        labelcolor='white', 
        fontsize=9,
        bbox_to_anchor=(0.98, 0.98)
    )

    # X-as & Titel
    ax_hr.set_xlabel('Verstreken Tijd (minuten)', fontweight='bold', color=KLEUREN_PALETTE.get('text', '#FFF'))
    fig.suptitle('ETAPPE PERFORMANCE: Herstelhartslag (HRR)', fontsize=14, fontweight='bold', y=0.96, color=KLEUREN_PALETTE.get('text', '#FFF'))
    
    plt.tight_layout()
    return fig, df_pauses

In [0]:
# Het FIT-bestand van etappe 3
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260914_stubaier_hohenweg_etappe_3.fit"

# 1. Inlezen & Opschonen via de utils
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)
df_active = calculate_elapsed_time(df_clean)

# 2. Genereer de grafiek en haal de KPI op
fig, df_stops = analyze_heartraterecovery(df_clean, min_stop_sec=30, hr_drop_window=60)

# 3. Exporteer naar landscape formaat voor Instagram
export_for_instagram(fig, filename="20260914_heart_rate_recovery", format_type='landscape')